# Final Project: BERT for Palestinian POS Tagging

This notebook follows the class workflow but extends Curras with the Palestinian Digital Campus Corpus: 244 POS-annotated sentences about university computing, programming, networks, and machine-learning projects. It also reproduces the controlled Curras-only baseline and the independent Palestinian challenge-set evaluation.

## 1. Install the class-notebook dependencies

In [ ]:
!pip -q install torch transformers datasets scikit-learn accelerate gdown

## 2. Get the professor-provided Curras split
The Drive folder is the same public folder referenced in the original class notebook.

In [ ]:
!gdown --folder 'https://drive.google.com/drive/folders/1azGC6MLbkB3FTt5uxK1XFFdpY7X2DYsD?usp=sharing' --output data/Curras

## 3. Prepare the corpus
When this notebook is opened from the repository, keep it in the same folder as the project scripts. The first script deterministically recreates the annotated extension and combined Curras files.

In [ ]:
!python build_digital_campus_corpus.py

## 4. Fine-tune AraBERT
The experiment trains `aubmindlab/bert-base-arabertv02-twitter` on Curras plus the new training split. The original Curras test set is left unchanged, and a second test is run on the held-out digital-campus sentences.

In [ ]:
!python train_and_evaluate.py --epochs 2 --batch-size 12

## 5. Train the controlled Curras-only baseline
This run keeps the checkpoint, seed, labels, epochs, batch size, learning rate, and test sets fixed. Only the training corpus changes. Its outputs use separate filenames, so the extended model is not overwritten.

In [ ]:
!python train_and_evaluate.py --training-data curras-only --epochs 2 --batch-size 12

## 6. Evaluate both models on the challenge set

In [ ]:
!python build_challenge_set.py
!python evaluate_challenge_set.py
!python evaluate_challenge_set.py --model-dir final_model_curras_only --output-stem challenge_curras_only

## 7. Display the saved evaluation

In [ ]:
import json
from pathlib import Path
for filename in ['results.json', 'results_curras_only.json']:
    results = json.loads(Path(filename).read_text(encoding='utf-8'))
    print(f'\n{filename}:')
    for name, metrics in results['evaluation'].items():
        print(name, metrics)

comparison = json.loads(Path('baseline_comparison.json').read_text(encoding='utf-8'))
print('\nControlled comparison:')
print(json.dumps(comparison, indent=2))